In [1]:
from common import *
from database import run_query
from scipy import stats
import numpy as np

order_f = run_query("SELECT * FROM order_features")
reviews = run_query("SELECT order_id, review_score FROM order_reviews")
order_rev = order_f.merge(reviews, on="order_id", how="inner")

In [2]:
df = order_rev.dropna(subset=["delivery_delay_days", "review_score"])
on_time = df.loc[df["is_late"] == 0, "review_score"]
late = df.loc[df["is_late"] == 1, "review_score"]

print(f"On-time: n={len(on_time)}, mean={on_time.mean():.3f}")
print(f"Late:    n={len(late)}, mean={late.mean():.3f}")

# Assumption check: equal variances? (decides which t-test variant to use)
lev_stat, lev_p = stats.levene(on_time, late)
print(f"\nLevene's test p-value: {lev_p:.4f}")

# Welch's t-test (safer default — doesn't assume equal variances)
t_stat, t_p = stats.ttest_ind(late, on_time, equal_var=False)
print(f"\nWelch's T-test: t={t_stat:.3f}, p={t_p:.6f}")

# Robustness check since review_score is ordinal, not continuous
u_stat, u_p = stats.mannwhitneyu(late, on_time, alternative="less")
print(f"Mann-Whitney U (robustness check): p={u_p:.6f}")

On-time: n=88168, mean=4.294
Late:    n=7662, mean=2.565

Levene's test p-value: 0.0000

Welch's T-test: t=-89.412, p=0.000000
Mann-Whitney U (robustness check): p=0.000000


In [3]:
items = run_query("SELECT order_id, product_id, price FROM order_items")
products = run_query("""
    SELECT p.product_id, t.product_category_name_english AS category
    FROM products p JOIN product_category_translation t
    ON t.product_category_name = p.product_category_name
""")
items_cat = items.merge(products, on="product_id")

top10 = items_cat["category"].value_counts().head(10).index
subset = items_cat[items_cat["category"].isin(top10)]
groups = [g["price"].values for _, g in subset.groupby("category")]

# Assumption check: normality (Shapiro on a sample, since full data is too large)
for name, g in subset.groupby("category"):
    sample = g["price"].sample(min(500, len(g)), random_state=42)
    _, p = stats.shapiro(sample)
    print(f"{name:25s} Shapiro p={p:.4f}")

f_stat, f_p = stats.f_oneway(*groups)
print(f"\nOne-Way ANOVA: F={f_stat:.3f}, p={f_p:.6f}")

# Price is right-skewed (confirmed in EDA) -> log-transform and re-test
subset["log_price"] = np.log1p(subset["price"])
log_groups = [g["log_price"].values for _, g in subset.groupby("category")]
f_stat_log, f_p_log = stats.f_oneway(*log_groups)
print(f"ANOVA on log(price): F={f_stat_log:.3f}, p={f_p_log:.6f}")

# Non-parametric robustness check (doesn't assume normality at all)
h_stat, h_p = stats.kruskal(*groups)
print(f"Kruskal-Wallis: H={h_stat:.3f}, p={h_p:.6f}")

auto                      Shapiro p=0.0000
bed_bath_table            Shapiro p=0.0000
computers_accessories     Shapiro p=0.0000
furniture_decor           Shapiro p=0.0000
garden_tools              Shapiro p=0.0000
health_beauty             Shapiro p=0.0000
housewares                Shapiro p=0.0000
sports_leisure            Shapiro p=0.0000
telephony                 Shapiro p=0.0000
watches_gifts             Shapiro p=0.0000

One-Way ANOVA: F=310.299, p=0.000000
ANOVA on log(price): F=655.244, p=0.000000
Kruskal-Wallis: H=5522.611, p=0.000000


In [4]:
payments = run_query("SELECT order_id, payment_type FROM order_payments")
orders_status = run_query("SELECT order_id, order_status FROM orders")
pay_status = payments.merge(orders_status, on="order_id")

# Group rare statuses so expected cell counts aren't tiny (chi-square assumption)
status_counts = pay_status["order_status"].value_counts()
print(status_counts)

rare_statuses = status_counts[status_counts < 100].index
pay_status["status_grouped"] = pay_status["order_status"].where(
    ~pay_status["order_status"].isin(rare_statuses), "other"
)

contingency = pd.crosstab(pay_status["payment_type"], pay_status["status_grouped"])
print(contingency)

chi2, chi_p, dof, expected = stats.chi2_contingency(contingency)
print(f"\nChi-Square: chi2={chi2:.3f}, dof={dof}, p={chi_p:.6f}")

# Assumption check: no more than 20% of expected cells should be < 5
small_cells = (expected < 5).sum() / expected.size
print(f"% of expected cells < 5: {small_cells:.1%}")

order_status
delivered      100754
shipped          1166
canceled          661
unavailable       649
invoiced          325
processing        319
created             5
approved            2
Name: count, dtype: int64
status_grouped  canceled  delivered  invoiced  other  processing  shipped  \
payment_type                                                                
boleto                95      19191        67      2          70      209   
credit_card          444      74584       239      5         224      851   
debit_card             7       1486         6      0           2       22   
voucher              115       5493        13      0          23       84   

status_grouped  unavailable  
payment_type                 
boleto                  150  
credit_card             446  
debit_card                6  
voucher                  47  

Chi-Square: chi2=209.989, dof=18, p=0.000000
% of expected cells < 5: 17.9%


In [5]:
print(contingency.div(contingency.sum(axis=1), axis=0).round(4) * 100)

status_grouped  canceled  delivered  invoiced  other  processing  shipped  \
payment_type                                                                
boleto              0.48      97.00      0.34   0.01        0.35     1.06   
credit_card         0.58      97.12      0.31   0.01        0.29     1.11   
debit_card          0.46      97.19      0.39   0.00        0.13     1.44   
voucher             1.99      95.12      0.23   0.00        0.40     1.45   

status_grouped  unavailable  
payment_type                 
boleto                 0.76  
credit_card            0.58  
debit_card             0.39  
voucher                0.81  


In [6]:
subset.groupby("category")["price"].mean().sort_values()

category
telephony                 71.213978
furniture_decor           87.564494
housewares                90.788148
bed_bath_table            93.296327
garden_tools             111.630196
sports_leisure           114.344285
computers_accessories    116.513903
health_beauty            130.163531
auto                     139.957523
watches_gifts            201.135984
Name: price, dtype: float64